# Bank Loan Portfolio Analysis & Risk Insights
**Tools:** Python, Pandas, Matplotlib

> This notebook uses a synthetic dataset for educational/portfolio purposes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/loan_data.csv')
df.head()

In [ ]:
df.info()

print('Missing values:')
print(df.isna().sum())

print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
total_applications = len(df)
approved = (df['Loan_Status'] == 'Approved').sum()
approval_rate = approved / total_applications * 100
approved_value = df.loc[df['Loan_Status']=='Approved', 'Loan_Amount_INR'].sum()
default_rate = ((df['Default_Flag']=='Yes') & (df['Loan_Status']=='Approved')).sum() / approved * 100

pd.DataFrame({
    'Metric':['Total Applications','Approved Applications','Approval Rate %','Approved Loan Value','Default Rate %'],
    'Value':[total_applications,approved,round(approval_rate,2),approved_value,round(default_rate,2)]
})

In [ ]:
employment_analysis = df.groupby('Employment_Type').agg(
    Applications=('Loan_ID','count'),
    Approval_Rate=('Loan_Status', lambda x: (x=='Approved').mean()*100),
    Avg_Loan=('Loan_Amount_INR','mean')
).sort_values('Approval_Rate', ascending=False)
employment_analysis.round(2)

In [ ]:
credit_band = pd.cut(df['Credit_Score'], bins=[0,599,699,749,850], labels=['Below 600','600-699','700-749','750+'])
credit_analysis = df.assign(Credit_Band=credit_band).groupby('Credit_Band', observed=False).agg(
    Applications=('Loan_ID','count'),
    Approval_Rate=('Loan_Status', lambda x: (x=='Approved').mean()*100),
    Default_Rate=('Default_Flag', lambda x: (x=='Yes').mean()*100)
)
credit_analysis.round(2)

In [ ]:
purpose = df.groupby('Loan_Purpose').agg(
    Applications=('Loan_ID','count'),
    Approved_Loan_Value=('Loan_Amount_INR', lambda x: x[df.loc[x.index,'Loan_Status']=='Approved'].sum()),
    Approval_Rate=('Loan_Status', lambda x: (x=='Approved').mean()*100)
).sort_values('Approved_Loan_Value', ascending=False)
purpose.round(2)

In [ ]:
plt.figure(figsize=(8,5))
employment_analysis['Approval_Rate'].plot(kind='bar')
plt.title('Approval Rate by Employment Type')
plt.ylabel('Approval Rate (%)')
plt.xlabel('Employment Type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
purpose['Approved_Loan_Value'].plot(kind='bar')
plt.title('Approved Loan Value by Loan Purpose')
plt.ylabel('Approved Loan Value (INR)')
plt.xlabel('Loan Purpose')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## Interpretation
- Compare approval conversion across employment categories.
- Examine credit-score bands for approval/default patterns.
- Use loan-purpose exposure to identify where portfolio value is concentrated.
- Treat all findings as illustrative because the dataset is synthetic.